In [47]:
"""
calcular_metricas.py
====================
Calcula as métricas fundamentalistas trimestrais para todas as empresas:

    Lucratividade : ll_positivo, roe, margem_liquida
    Crescimento   : cagr_lucro_5a, cagr_receita_5a  (comparação ano a ano)
    Endividamento : dl_ebitda  (já calculado no itr_ranking)
    Valuation     : pl (P/L)
    Selic         : baixada do Banco Central para comparação com ROE

Uso:
    from calcular_metricas import calcular_metricas
    df_metricas = calcular_metricas(historico_completo)
"""

import requests
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# 1. Taxa Selic histórica (Banco Central do Brasil)
# ---------------------------------------------------------------------------

from datetime import datetime, timedelta

def _baixar_selic() -> pd.Series:
    inicio = (datetime.today() - timedelta(days=365 * 10)).strftime("%d/%m/%Y")
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial={inicio}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        df = pd.DataFrame(r.json())
        df["data"]  = pd.to_datetime(df["data"], format="%d/%m/%Y")
        df["valor"] = pd.to_numeric(df["valor"], errors="coerce") / 100
        df = df.set_index("data")["valor"].sort_index()
        print(f"  Selic: {len(df):,} pontos ({df.index.min().date()} → {df.index.max().date()})")
        return df
    except Exception as e:
        print(f"  ⚠ Selic indisponível: {e}")
        return pd.Series(dtype=float)


def _selic_anual_por_data(selic: pd.Series, datas: pd.DatetimeIndex) -> pd.Series:
    """Para cada data, retorna a Selic mais recente disponível."""
    if selic.empty:
        return pd.Series(np.nan, index=datas)
    return pd.Series(
        [selic.asof(d) if d >= selic.index[0] else np.nan for d in datas],
        index=datas,
    )


# ---------------------------------------------------------------------------
# 2. Annualização: garante que LL e Receita estejam em base anual
#    (já feito para EBIT no itr_ranking; replicamos aqui para ll e receita)
# ---------------------------------------------------------------------------

def _anualizar_serie(df: pd.DataFrame, col: str) -> pd.Series:
    """
    Anualiza uma coluna YTD usando DT_INI_EXERC e DT_FIM_EXERC.
    Retorna série com os valores anualizados.
    """
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index)

    if "DT_INI_EXERC" in df.columns and "DT_FIM_EXERC" in df.columns:
        meses = (
            (df["DT_FIM_EXERC"] - df["DT_INI_EXERC"])
            / pd.Timedelta(days=30.4375)
        ).clip(lower=1).round().fillna(12)
        fator = (12 / meses).clip(upper=4)
    else:
        fator = pd.Series(1.0, index=df.index)

    return df[col] * fator


# ---------------------------------------------------------------------------
# 3. CAGR — calculado sobre valores anuais (DFP, mês 12)
# ---------------------------------------------------------------------------

def _calcular_cagr(df: pd.DataFrame, col: str, anos: int = 5) -> pd.Series:
    """
    Calcula CAGR usando apenas dados de dezembro (DFP = ano completo).
    Nunca faz merge no df original — usa lookup por dicionário.
    """
    # Extrai só o necessário: empresa, ano, valor de dezembro
    mask_dez = df["DT_FIM_EXERC"].dt.month == 12
    anual = (
        df[mask_dez]
        .assign(_ano=df["DT_FIM_EXERC"].dt.year)
        .groupby(["CNPJ_CIA", "_ano"])[col]
        .mean()                         # média caso haja duplicata
        .reset_index()
    )

    if anual.empty:
        print(f"  ⚠ Sem datas de dezembro para '{col}'")
        return pd.Series(np.nan, index=df.index)

    print(f"  '{col}': {anual['CNPJ_CIA'].nunique()} empresas com dezembro")

    # Monta dicionário {(CNPJ, ano_fim): cagr}
    # Para cada empresa, percorre seus anos disponíveis
    cagr_dict = {}
    for cnpj, grupo in anual.groupby("CNPJ_CIA"):
        vals = grupo.set_index("_ano")[col]   # Series indexada por ano
        for ano_fim in vals.index:
            ano_ini = ano_fim - anos
            if ano_ini not in vals.index:
                continue
            v_fim = vals[ano_fim]
            v_ini = vals[ano_ini]
            if pd.isna(v_fim) or pd.isna(v_ini) or v_ini <= 0 or v_fim <= 0:
                continue
            cagr_dict[(cnpj, ano_fim)] = (v_fim / v_ini) ** (1 / anos) - 1

    print(f"  '{col}': {len(cagr_dict)} CAGRs calculados")

    # Mapeia de volta ao df original por (CNPJ, ano) — sem merge, sem iterrows
    anos_df = df["DT_FIM_EXERC"].dt.year
    result  = [
        cagr_dict.get((cnpj, ano), np.nan)
        for cnpj, ano in zip(df["CNPJ_CIA"], anos_df)
    ]
    return pd.Series(result, index=df.index)   # índice original preservado
# ---------------------------------------------------------------------------
# 4. P/L
# ---------------------------------------------------------------------------

def _calcular_pl(df: pd.DataFrame) -> pd.Series:
    """
    P/L = Market Cap / Lucro Líquido Anualizado.
    Retorna NaN se LL <= 0 (empresa com prejuízo ou break-even).
    """
    ll_anual = _anualizar_serie(df, "ll")
    pl = np.where(
        ll_anual.notna() & (ll_anual > 0) & df["market_cap"].notna(),
        df["market_cap"] / ll_anual,
        np.nan,
    )
    return pd.Series(pl, index=df.index)


# ---------------------------------------------------------------------------
# 5. Pipeline principal
# ---------------------------------------------------------------------------

def calcular_metricas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recebe o DataFrame completo (saída de adicionar_ev) e adiciona:

        ll_positivo      : True se lucro líquido anualizado > 0
        roe              : Lucro Líquido Anualizado / PL  (em %)
        margem_liquida   : LL Anualizado / Receita Anualizada  (em %)
        cagr_lucro_5a    : CAGR do Lucro nos últimos 5 anos (em %)
        cagr_receita_5a  : CAGR da Receita nos últimos 5 anos (em %)
        dl_ebitda        : Dívida Líquida / EBITDA  (já no df, garante presença)
        pl               : P/L  (Market Cap / LL Anualizado)
        selic            : Meta Selic na data do trimestre  (em %)
        roe_vs_selic     : ROE - Selic  (positivo = bate a Selic)

    Critérios de descarte também são adicionados como flags booleanas:
        descartar_prejuizo    : LL <= 0
        descartar_pl_alto     : P/L > 15
        descartar_dl_ebitda   : DL/EBITDA > 3 (exceto setores de infraestrutura)
        descartar_cagr_lucro  : CAGR lucro 5a negativo
    """
    df = df.copy()
    df["DT_FIM_EXERC"]  = pd.to_datetime(df["DT_FIM_EXERC"])
    if "DT_INI_EXERC" in df.columns:
        df["DT_INI_EXERC"] = pd.to_datetime(df["DT_INI_EXERC"])

    print("Calculando métricas fundamentalistas...")

    # --- Selic histórica ---
    print("  Baixando Selic (BCB)...")
    selic = _baixar_selic()
    datas_unicas = pd.DatetimeIndex(df["DT_FIM_EXERC"].dropna().unique())
    mapa_selic   = _selic_anual_por_data(selic, datas_unicas)
    df["selic"]  = df["DT_FIM_EXERC"].map(mapa_selic)

    # --- Valores anualizados ---
    ll_anual      = _anualizar_serie(df, "ll")
    receita_anual = _anualizar_serie(df, "receita")

    # --- Lucratividade ---
    df["ll_anualizado"]  = ll_anual
    df["ll_positivo"]    = ll_anual > 0

    pl_val = df.get("pl", pd.Series(np.nan, index=df.index))   # PL patrimonial
    df["roe"] = np.where(
        pl_val.abs() > 1e-6,
        ll_anual / pl_val,
        np.nan,
    )

    df["margem_liquida"] = np.where(
        receita_anual.abs() > 1e-6,
        ll_anual / receita_anual,
        np.nan,
    )

    df["roe_vs_selic"] = df["roe"] - df["selic"]

    # --- Crescimento (CAGR 5 anos) ---
    print("  Calculando CAGR de lucro e receita (5 anos)...")
    df["ll_anual_para_cagr"]      = ll_anual       # coluna temporária
    df["receita_anual_para_cagr"] = receita_anual

    df["cagr_lucro_5a"]   = _calcular_cagr(df, "ll_anual_para_cagr",      anos=5)
    df["cagr_receita_5a"] = _calcular_cagr(df, "receita_anual_para_cagr", anos=5)

    df = df.drop(columns=["ll_anual_para_cagr", "receita_anual_para_cagr"],
                 errors="ignore")

    # --- Valuation ---
    df["p_l"] = _calcular_pl(df)

    # --- Flags de descarte ---
    df["descartar_prejuizo"]   = ~df["ll_positivo"].fillna(False)

    df["descartar_pl_alto"]    = df["p_l"] > 15

    dl_ebitda = df.get("dl_ebitda", pd.Series(np.nan, index=df.index))
    df["descartar_dl_ebitda"]  = dl_ebitda > 3

    df["descartar_cagr_lucro"] = df["cagr_lucro_5a"] < 0

    # --- Converte ratios para % ---
    for col in ["roe", "margem_liquida", "cagr_lucro_5a",
                "cagr_receita_5a", "selic", "roe_vs_selic"]:
        if col in df.columns:
            df[col] = (df[col] * 100).round(2)

    df["p_l"] = df["p_l"].round(2)

    print("  ✅ Métricas calculadas")
    _resumo(df)
    return df


def _resumo(df: pd.DataFrame) -> None:
    """Imprime um resumo das flags de descarte no último trimestre."""
    ultimo = df["DT_FIM_EXERC"].max()
    recente = df[df["DT_FIM_EXERC"] == ultimo]
    total   = len(recente)
    print(f"\n  Resumo no trimestre mais recente ({str(ultimo)[:10]}, {total} empresas):")

    flags = {
        "  Prejuízo":        "descartar_prejuizo",
        "  P/L > 15":        "descartar_pl_alto",
        "  DL/EBITDA > 3":   "descartar_dl_ebitda",
        "  CAGR lucro < 0":  "descartar_cagr_lucro",
    }
    for label, col in flags.items():
        if col in recente.columns:
            n = recente[col].sum()
            print(f"    {label}: {n:,} ({100*n/total:.0f}%)")


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------
# if __name__ == "__main__":
#     # Supondo historico_completo já gerado por adicionar_ev()
#     df_metricas = calcular_metricas(historico_completo)

#     # Empresas aprovadas no último trimestre
#     ultimo = df_metricas["DT_FIM_EXERC"].max()
#     aprovadas = df_metricas[
#         (df_metricas["DT_FIM_EXERC"] == ultimo)
#         & (~df_metricas["descartar_prejuizo"])
#         & (~df_metricas["descartar_pl_alto"])
#         & (~df_metricas["descartar_dl_ebitda"])
#         & (~df_metricas["descartar_cagr_lucro"].fillna(True))
#         & (df_metricas["margem_liquida"] > 13)
#         & (df_metricas["roe_vs_selic"] > -2)
#     ].sort_values("rank_magic_setor", na_position="last")

#     print(f"\nEmpresas aprovadas em todos os critérios: {len(aprovadas)}")
#     print(aprovadas[["DENOM_CIA", "ticker", "roe", "margem_liquida",
#                       "p_l", "dl_ebitda", "cagr_lucro_5a"]].to_string())

#     df_metricas.to_csv("metricas_completas.csv", index=False, encoding="utf-8-sig")

In [27]:
"""
itr_ranking.py
==============
Baixa ITR + DFP da CVM, calcula EV/EBIT e ROIC anualizados
de TODAS as empresas não-financeiras e retorna ranking por trimestre.

Correções aplicadas:
    1. DFP incluído → cobre Q4 (dezembro) para todas as empresas
    2. EBIT anualizado via DT_INI_EXERC → elimina distorção entre trimestres
    3. Setor financeiro filtrado → bancos/seguradoras excluídos
"""

import requests
import numpy as np
import pandas as pd
from io import BytesIO
from zipfile import ZipFile

# ---------------------------------------------------------------------------
# Contas CVM
# ---------------------------------------------------------------------------
CONTAS_DRE = {"ebit": "3.05", "ll": "3.11", "ir_csll": "3.08", "receita": "3.01"}
CONTAS_BPA = {"caixa": "1.01.01", "aplic_cp": "1.01.02"}
CONTAS_BPP = {"divida_cp": "2.01.04", "divida_lp": "2.02.01", "pl": "2.03"}

ITR_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS"
DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS"
CAD_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"

# Setores financeiros a excluir (campo SETOR_ATIV do cadastro CVM)
SETORES_FINANCEIROS = {
    "Intermediários Financeiros",
    "Previdência e Seguros",
    "Auxiliares Financeiros",
    "Outros Intermediários Financeiros e Serviços Relacionados",
}

# ---------------------------------------------------------------------------
# 1. Download — ITR + DFP
# ---------------------------------------------------------------------------

def _ler_csv_do_zip(content: bytes, nome_csv: str) -> pd.DataFrame | None:
    try:
        with ZipFile(BytesIO(content)) as z:
            if nome_csv not in z.namelist():
                return None
            return pd.read_csv(
                z.open(nome_csv), sep=";", encoding="latin1", dtype=str
            )
    except Exception as e:
        print(f"    ✗ Erro ao ler {nome_csv}: {e}")
        return None


def baixar_dados(ano: int) -> dict[str, pd.DataFrame]:
    """
    Baixa ITR (Q1–Q3) e DFP (Q4) do ano e retorna {'dre', 'bpa', 'bpp'}.
    ITR cobre março/junho/setembro; DFP cobre dezembro (ano fiscal padrão).
    """
    frames: dict[str, list] = {"dre": [], "bpa": [], "bpp": [], "dfc": []}
    tipos  = {"dre": "DRE_con", "bpa": "BPA_con", "bpp": "BPP_con", "dfc": "DFC_MI_con"}

    for fonte, base_url in [("ITR", ITR_URL), ("DFP", DFP_URL)]:
        prefixo = "itr" if fonte == "ITR" else "dfp"
        zip_url = f"{base_url}/{prefixo}_cia_aberta_{ano}.zip"
        print(f"  Baixando {fonte} {ano}...", end=" ", flush=True)
        try:
            r = requests.get(zip_url, timeout=120)
            r.raise_for_status()
            print("✓")
        except Exception as e:
            print(f"✗ ({e})")
            continue

        for chave, sufixo in tipos.items():
            nome_csv = f"{prefixo}_cia_aberta_{sufixo}_{ano}.csv"
            df = _ler_csv_do_zip(r.content, nome_csv)
            if df is not None:
                df["_fonte"] = fonte   # marca origem para deduplicação posterior
                frames[chave].append(df)

    return {
        chave: pd.concat(dfs, ignore_index=True)
        for chave, dfs in frames.items()
        if dfs
    }


# ---------------------------------------------------------------------------
# 2. CNPJs do setor financeiro (para filtrar)
# ---------------------------------------------------------------------------

def _cnpjs_financeiros() -> set[str]:
    """Retorna conjunto de CNPJs de empresas em setores financeiros."""
    try:
        cad = pd.read_csv(CAD_URL, sep=";", encoding="latin1", dtype=str)
        cad.columns = cad.columns.str.strip()
        cad["CNPJ_CIA"]   = cad["CNPJ_CIA"].str.strip()
        cad["SETOR_ATIV"] = cad["SETOR_ATIV"].str.strip()
        fin = cad[cad["SETOR_ATIV"].isin(SETORES_FINANCEIROS)]
        print(f"  Setor financeiro: {len(fin):,} empresas serão excluídas")
        return set(fin["CNPJ_CIA"].unique())
    except Exception as e:
        print(f"  ⚠ Não foi possível filtrar setor financeiro: {e}")
        return set()


# ---------------------------------------------------------------------------
# 3. Limpeza
# ---------------------------------------------------------------------------

def _limpar(df: pd.DataFrame, cnpjs_financeiros: set[str]) -> pd.DataFrame:
    df = df.copy()
    df = df[~df["CNPJ_CIA"].isin(cnpjs_financeiros)]
    df = df[df["ORDEM_EXERC"] == "ÚLTIMO"]

    df["DT_FIM_EXERC_dt"] = pd.to_datetime(df["DT_FIM_EXERC"], errors="coerce")
    mask_itr_dez = (df["_fonte"] == "ITR") & (df["DT_FIM_EXERC_dt"].dt.month == 12)
    df = df[~mask_itr_dez].drop(columns=["DT_FIM_EXERC_dt"])

    tem_cons = df[df["GRUPO_DFP"].str.contains("Consolidado", na=False)]["CNPJ_CIA"].unique()
    mask_cons = df["GRUPO_DFP"].str.contains("Consolidado", na=False)
    mask_ind  = ~df["CNPJ_CIA"].isin(tem_cons)
    df = df[mask_cons | mask_ind]

    df["VERSAO"] = pd.to_numeric(df["VERSAO"], errors="coerce")
    df = (
        df.sort_values("VERSAO", ascending=False)
          .drop_duplicates(subset=["CNPJ_CIA", "DT_FIM_EXERC", "CD_CONTA"], keep="first")
    )

    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"], errors="coerce")
    df["VL_CONTA"]     = pd.to_numeric(df["VL_CONTA"], errors="coerce")

    # ← a linha problemática, agora protegida
    if "DT_INI_EXERC" in df.columns:
        df["DT_INI_EXERC"] = pd.to_datetime(df["DT_INI_EXERC"], errors="coerce")

    mask_mil = df["ESCALA_MOEDA"].str.upper().str.contains("MIL", na=False)
    df.loc[mask_mil, "VL_CONTA"] *= 1000

    return df.drop(columns=["_fonte"], errors="ignore")


# ---------------------------------------------------------------------------
# 4. Pivotamento
# ---------------------------------------------------------------------------

def _pivotar_dre(df: pd.DataFrame) -> pd.DataFrame:
    """
    DRE: inclui DT_INI_EXERC no índice para calcular período real depois.
    """
    mapa_inv = {v: k for k, v in CONTAS_DRE.items()}
    sub = df[df["CD_CONTA"].isin(CONTAS_DRE.values())].copy()
    sub["nome_conta"] = sub["CD_CONTA"].map(mapa_inv)

    piv = sub.pivot_table(
        index=["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC", "DT_INI_EXERC"],
        columns="nome_conta",
        values="VL_CONTA",
        aggfunc="first",
    ).reset_index()
    piv.columns.name = None
    return piv


def _pivotar_balanco(df: pd.DataFrame, contas: dict) -> pd.DataFrame:
    """Balanço: não precisa de DT_INI_EXERC (valores pontuais)."""
    mapa_inv = {v: k for k, v in contas.items()}
    sub = df[df["CD_CONTA"].isin(contas.values())].copy()
    sub["nome_conta"] = sub["CD_CONTA"].map(mapa_inv)

    piv = sub.pivot_table(
        index=["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"],
        columns="nome_conta",
        values="VL_CONTA",
        aggfunc="first",
    ).reset_index()
    piv.columns.name = None
    return piv


# ---------------------------------------------------------------------------
# 5. Cálculo com EBIT anualizado
# ---------------------------------------------------------------------------

def _calcular(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula ROIC com EBIT anualizado.

    Anualização:
        meses_reportados = (DT_FIM_EXERC - DT_INI_EXERC) em meses
        ebit_anualizado  = ebit_ytd * (12 / meses_reportados)

    Exemplos:
        Q1 (3 meses):  fator = 12/3  = 4.0
        Q2 (6 meses):  fator = 12/6  = 2.0
        Q3 (9 meses):  fator = 12/9  = 1.33
        Q4/DFP (12m):  fator = 12/12 = 1.0  (sem alteração)
    """
    df = df.copy()

    # --- Anualização do EBIT ---
    meses = (
        (df["DT_FIM_EXERC"] - df["DT_INI_EXERC"])
        / pd.Timedelta(days=30.4375)     # média de dias por mês
    ).clip(lower=1).round()

    fator = (12 / meses).clip(upper=4)  # máximo 4x (evita distorção em períodos < 1 mês)

    ebit_raw = df.get("ebit", pd.Series(np.nan, index=df.index))
    df["ebit_anualizado"] = ebit_raw * fator
    df["meses_periodo"]   = meses       # coluna auxiliar para auditoria

    ebit = df["ebit_anualizado"]        # usa anualizado em todos os cálculos
    ll   = df.get("ll",      pd.Series(np.nan, index=df.index))
    ir   = df.get("ir_csll", pd.Series(0,      index=df.index)).fillna(0).abs()

    # --- Balanço ---
    dispon    = df.get("caixa",    pd.Series(0, index=df.index)).fillna(0) \
              + df.get("aplic_cp", pd.Series(0, index=df.index)).fillna(0)
    div_bruta = df.get("divida_cp", pd.Series(0, index=df.index)).fillna(0) \
              + df.get("divida_lp", pd.Series(0, index=df.index)).fillna(0)
    div_liq   = div_bruta - dispon
    pl        = df.get("pl", pd.Series(np.nan, index=df.index))

    df["disponibilidades"] = dispon
    df["divida_bruta"]     = div_bruta
    df["divida_liquida"]   = div_liq

    # --- Alíquota efetiva ---
    ll_anualizado = ll * fator
    base_ir = ll_anualizado.abs() + ir * fator
    aliq = np.where(base_ir > 0, (ir * fator) / base_ir, 0.34)
    df["aliquota_efetiva"] = np.clip(aliq, 0, 0.50)

    # --- NOPAT e Capital Investido ---
    df["nopat"]             = ebit * (1 - df["aliquota_efetiva"])
    cap_inv                 = pl.fillna(0) + div_liq
    df["capital_investido"] = cap_inv

    # --- ROIC ---
    df["roic"] = np.where(
        cap_inv.abs() > 1e-6,
        df["nopat"] / cap_inv,
        np.nan,
    )

    # D&A anualizado e EBITDA
    da_raw = df.get("da", pd.Series(0, index=df.index)).fillna(0)
    df["da_anualizado"] = da_raw * fator
    df["ebitda"]        = ebit + df["da_anualizado"]

    # Dívida Líquida / EBITDA
    df["dl_ebitda"] = np.where(
        df["ebitda"].abs() > 1e-6,
        div_liq / df["ebitda"],
        np.nan,
    )

    # Colunas de mercado — preenchidas por adicionar_ev.py
    df["market_cap"] = np.nan
    df["ev"]         = np.nan
    df["ev_ebit"]    = np.nan

    return df


# ---------------------------------------------------------------------------
# 6. Ranking por trimestre
# ---------------------------------------------------------------------------

def _rankear(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ROIC: maior é melhor → rank 1 = maior ROIC
    df["rank_roic"] = (
        df[df["roic"] > 0]        # exclui ROIC negativos do ranking
          .groupby("DT_FIM_EXERC")["roic"]
          .rank(ascending=False, method="min", na_option="bottom")
    ).reindex(df.index).astype("Int64")

    return df.sort_values(["DT_FIM_EXERC", "rank_roic"])


# ---------------------------------------------------------------------------
# 6b. Extração de D&A do DFC
# ---------------------------------------------------------------------------

def _extrair_da(dfc: pd.DataFrame) -> pd.DataFrame:
    """
    Extrai Depreciação & Amortização do DFC Método Indireto.
    Busca por DS_CONTA que contenha 'deprecia' ou 'amortiza' (case-insensitive).
    Soma todos os valores encontrados por empresa/trimestre (D&A total).
    """
    if dfc.empty or "DS_CONTA" not in dfc.columns:
        return pd.DataFrame()

    mask_da = dfc["DS_CONTA"].str.lower().str.contains(
        "deprecia|amortiza", na=False
    )
    sub = dfc[mask_da].copy()
    if sub.empty:
        return pd.DataFrame()

    # D&A no DFC aparece como valor positivo (adição de volta ao lucro)
    sub["VL_CONTA"] = sub["VL_CONTA"].abs()

    da = (
        sub.groupby(["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"])["VL_CONTA"]
           .sum()
           .reset_index()
           .rename(columns={"VL_CONTA": "da"})
    )
    return da


# ---------------------------------------------------------------------------
# 7. Pipeline principal
# ---------------------------------------------------------------------------

def processar_ano(ano: int) -> pd.DataFrame:
    """
    Pipeline completo para um ano.
    Baixa ITR (Q1–Q3) + DFP (Q4), filtra financeiras,
    anualiza EBIT e retorna ranking trimestral de todas as empresas.

    Para vários anos:
        frames = [processar_ano(a) for a in range(2019, 2025)]
        historico = pd.concat(frames, ignore_index=True)
    """
    print(f"\n{'='*55}")
    print(f"Processando {ano}")
    print(f"{'='*55}")

    # Download
    dados = baixar_dados(ano)
    if not dados:
        return pd.DataFrame()

    # CNPJs financeiros
    print("Filtrando setor financeiro...")
    cnpjs_fin = _cnpjs_financeiros()

    # Limpeza
    print("Limpando dados...")
    dre = _limpar(dados["dre"], cnpjs_fin)
    bpa = _limpar(dados["bpa"], cnpjs_fin)
    bpp = _limpar(dados["bpp"], cnpjs_fin)

    # Pivotamento
    print("Pivotando contas...")
    df_dre = _pivotar_dre(dre)
    df_bpa = _pivotar_balanco(bpa, CONTAS_BPA)
    df_bpp = _pivotar_balanco(bpp, CONTAS_BPP)

    # D&A do DFC: busca contas de depreciação/amortização por nome
    df_da = pd.DataFrame()
    if "dfc" in dados:
        dfc = _limpar(dados["dfc"], cnpjs_fin)
        df_da = _extrair_da(dfc)

    # Merge — DRE tem DT_INI_EXERC extra
    chave_bal = ["CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC"]
    base = (
        df_dre
        .merge(df_bpa, on=chave_bal, how="outer")
        .merge(df_bpp, on=chave_bal, how="outer")
    )
    if not df_da.empty:
        base = base.merge(df_da, on=chave_bal, how="left")

    trimestres = sorted(base["DT_FIM_EXERC"].dropna().unique())
    print(f"  {base['CNPJ_CIA'].nunique():,} empresas | "
          f"{len(trimestres)} datas: {[str(t)[:10] for t in trimestres]}")

    # Cálculo e ranking
    print("Calculando indicadores (EBIT anualizado)...")
    resultado = _calcular(base)
    resultado = _rankear(resultado)

    # Colunas finais
    cols = [
        "CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC", "DT_INI_EXERC", "meses_periodo",
        "receita", "ll", "ebit", "ebit_anualizado", "nopat",
        "da_anualizado", "ebitda",
        "disponibilidades", "divida_bruta", "divida_liquida",
        "pl", "capital_investido",
        "roic", "dl_ebitda", "rank_roic",
        "market_cap", "ev", "ev_ebit",
    ]
    cols_presentes = [c for c in cols if c in resultado.columns]
    resultado = resultado[cols_presentes]

    print(f"✅ {ano} concluído: {len(resultado):,} linhas")
    return resultado

In [28]:
"""
adicionar_ev.py
===============
Adiciona market_cap, ev, ev_ebit e rank_magic ao DataFrame do itr_ranking.
"""

import time
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from rapidfuzz import process, fuzz


# ---------------------------------------------------------------------------
# Mapeamento CNPJ → ticker via brapi.dev + fuzzy match
# ---------------------------------------------------------------------------

def _buscar_todos_tickers_brapi() -> pd.DataFrame:
    url = "https://brapi.dev/api/quote/list"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        stocks = r.json().get("stocks", [])
        df = pd.DataFrame(stocks)[["stock", "name"]].copy()
        df.columns = ["ticker", "nome"]
        df["ticker"] = df["ticker"].str.upper().str.strip()
        df["nome"]   = df["nome"].str.upper().str.strip()
        df = df[df["ticker"].str.match(r'^[A-Z]{4}\d{1,2}$')]
        print(f"  brapi.dev: {len(df):,} tickers carregados")
        return df.drop_duplicates("ticker").reset_index(drop=True)
    except Exception as e:
        print(f"  ✗ Erro brapi.dev: {e}")
        return pd.DataFrame(columns=["ticker", "nome"])


def _normalizar_nome(nome: str) -> str:
    remover = [
        "S.A.", "S/A", "SA", "LTDA", "LTDA.", "S.A", "/SA",
        "CIA.", "CIA", "COMPANHIA", "PARTICIPACOES", "PARTICIPAÇÕES",
        "HOLDING", "GROUP", "BRASIL", "DO BRASIL",
        "EM RECUPERACAO JUDICIAL", "EM LIQUIDACAO EXTRAJUDICIAL",
    ]
    nome = nome.upper()
    for t in remover:
        nome = nome.replace(t, "")
    return " ".join(nome.split())


def _baixar_mapa_cnpj_ticker(df_ranking: pd.DataFrame) -> pd.DataFrame:
    empresas = (
        df_ranking[["CNPJ_CIA", "DENOM_CIA"]]
        .drop_duplicates("CNPJ_CIA")
        .dropna(subset=["DENOM_CIA"])
        .copy()
    )
    print(f"  Empresas para mapear: {len(empresas):,}")

    df_brapi = _buscar_todos_tickers_brapi()
    if df_brapi.empty:
        return pd.DataFrame(columns=["CNPJ_CIA", "TCKR"])

    nomes_brapi_norm = df_brapi["nome"].apply(_normalizar_nome).tolist()
    tickers_brapi    = df_brapi["ticker"].tolist()

    mapa = {}
    for nome in empresas["DENOM_CIA"].unique():
        nome_norm = _normalizar_nome(nome)
        match = process.extractOne(
            nome_norm, nomes_brapi_norm, scorer=fuzz.token_sort_ratio
        )
        mapa[nome] = tickers_brapi[nomes_brapi_norm.index(match[0])] \
                     if match and match[1] >= 72 else None

    empresas["TCKR"] = empresas["DENOM_CIA"].map(mapa)
    encontrados = empresas["TCKR"].notna().sum()
    print(f"  Mapeadas: {encontrados:,}/{len(empresas):,} "
          f"({len(empresas)-encontrados:,} não listadas na B3 — esperado)")

    return empresas[["CNPJ_CIA", "TCKR"]].dropna(subset=["TCKR"])


# ---------------------------------------------------------------------------
# Preços históricos via yfinance
# ---------------------------------------------------------------------------

def _buscar_precos_ticker(ticker: str, datas) -> pd.DataFrame:
    datas_dt = pd.to_datetime(datas)
    try:
        t      = yf.Ticker(f"{ticker}.SA")
        shares = getattr(t.fast_info, "shares", None) or t.info.get("sharesOutstanding")
        if not shares:
            return pd.DataFrame()

        hist = t.history(
            start=datas_dt.min() - pd.DateOffset(days=15),
            end=datas_dt.max()   + pd.DateOffset(days=15),
            interval="1mo",
            auto_adjust=True,
        )
        if hist.empty:
            return pd.DataFrame()

        hist = hist[["Close"]].reset_index()
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None)

        rows = []
        for data in datas_dt:
            idx = (hist["Date"] - data).abs().idxmin()
            rows.append({
                "DT_FIM_EXERC": data,
                "preco":        hist.loc[idx, "Close"],
                "shares":       shares,
            })
        return pd.DataFrame(rows)
    except Exception:
        return pd.DataFrame()


# ---------------------------------------------------------------------------
# Pipeline principal
# ---------------------------------------------------------------------------

def adicionar_ev(df_ranking: pd.DataFrame, delay: float = 0.3) -> pd.DataFrame:
    """
    Adiciona market_cap, ev, ev_ebit e rank_magic ao DataFrame do itr_ranking.
    """
    df = df_ranking.copy()
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])

    # --- Mapeamento CNPJ → ticker ---
    print("Construindo mapeamento CNPJ → ticker...")
    mapa = _baixar_mapa_cnpj_ticker(df)
    if mapa.empty:
        print("✗ Sem mapeamento. Abortando.")
        return df

    df = df.merge(mapa, on="CNPJ_CIA", how="left")
    print(f"  {df['TCKR'].notna().sum():,} linhas com ticker | "
          f"{df['TCKR'].isna().sum():,} sem ticker")

    # --- Busca preços: um ticker de cada vez ---
    tickers_unicos = df["TCKR"].dropna().unique()
    print(f"\nBuscando preços para {len(tickers_unicos):,} tickers...")

    frames_precos = []
    for i, ticker in enumerate(tickers_unicos):
        datas = df.loc[df["TCKR"] == ticker, "DT_FIM_EXERC"].dropna().unique()
        df_p  = _buscar_precos_ticker(ticker, datas)
        if not df_p.empty:
            df_p["TCKR"] = ticker
            frames_precos.append(df_p)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(tickers_unicos)} ({len(frames_precos)} com dados)...")
        time.sleep(delay)

    print(f"  Preços obtidos: {len(frames_precos):,} tickers")

    if not frames_precos:
        print("✗ Nenhum preço obtido.")
        return df.drop(columns=["TCKR"], errors="ignore")

    # --- Merge de preços: substitui o loop linha a linha ---
    # Constrói um DataFrame único com todos os preços e faz merge por (TCKR, DT_FIM_EXERC)
    df_precos = pd.concat(frames_precos, ignore_index=True)
    df_precos["DT_FIM_EXERC"] = pd.to_datetime(df_precos["DT_FIM_EXERC"])

    df = df.merge(
        df_precos[["TCKR", "DT_FIM_EXERC", "preco", "shares"]],
        on=["TCKR", "DT_FIM_EXERC"],
        how="left",
    )

    # Mantém ticker como coluna nomeada antes de dropar TCKR
    df = df.rename(columns={"TCKR": "ticker"})

    # --- Indicadores de mercado ---
    df["market_cap"] = df["preco"] * df["shares"]
    df["ev"]         = df["market_cap"] + df["divida_liquida"]
    df["ev_ebit"]    = np.where(
        df["ebit"].notna() & (df["ebit"] != 0),
        df["ev"] / df["ebit"],
        np.nan,
    )

    # --- Rankings ---
    # EV/EBIT: só valores positivos fazem sentido (EBIT e EV positivos)
    df["rank_ev_ebit"] = (
        df[df["ev_ebit"] > 0]
          .groupby("DT_FIM_EXERC")["ev_ebit"]
          .rank(ascending=True, method="min", na_option="bottom")
    ).reindex(df.index).astype("Int64")

    df["rank_magic"] = np.where(
        df["rank_roic"].notna() & df["rank_ev_ebit"].notna(),
        df["rank_roic"].astype(float) + df["rank_ev_ebit"].astype(float),
        np.nan,
    )

    # --- Ordena por rank_magic (menor = melhor) dentro de cada trimestre ---
    df = df.sort_values(
        ["DT_FIM_EXERC", "rank_magic"],
        ascending=[True, True],
        na_position="last",
    ).reset_index(drop=True)

    # --- Reordena colunas: ticker logo após DENOM_CIA ---
    cols = df.columns.tolist()
    for col in ["ticker", "rank_roic", "rank_ev_ebit", "rank_magic"]:
        if col in cols:
            cols.remove(col)
    idx = cols.index("DENOM_CIA")
    cols = (
        cols[:idx + 1]
        + ["ticker"]
        + cols[idx + 1:]
        + ["rank_roic", "rank_ev_ebit", "rank_magic"]
    )
    df = df[[c for c in cols if c in df.columns]]

    preenchidos = df["market_cap"].notna().sum()
    print(f"\n✅ market_cap preenchido: {preenchidos:,}/{len(df):,} "
          f"({100*preenchidos/len(df):.1f}%)")
    return df

In [29]:
frames = [processar_ano(a) for a in range(2019, 2026)]
historico = pd.concat(frames, ignore_index=True)

# 2. Preços de mercado
historico_completo = adicionar_ev(historico)

# 3. Métricas fundamentalistas
df_final = calcular_metricas(historico_completo)


Processando 2019
  Baixando ITR 2019... ✓
  Baixando DFP 2019... ✓
Filtrando setor financeiro...
  Setor financeiro: 0 empresas serão excluídas
Limpando dados...
Pivotando contas...
  397 empresas | 9 datas: ['2019-01-01', '2019-02-28', '2019-03-31', '2019-05-31', '2019-06-30', '2019-08-31', '2019-09-30', '2019-11-30', '2019-12-31']
Calculando indicadores (EBIT anualizado)...
✅ 2019 concluído: 1,528 linhas

Processando 2020
  Baixando ITR 2020... ✓
  Baixando DFP 2020... ✓
Filtrando setor financeiro...
  Setor financeiro: 0 empresas serão excluídas
Limpando dados...
Pivotando contas...
  447 empresas | 8 datas: ['2020-02-29', '2020-03-31', '2020-05-31', '2020-06-30', '2020-08-31', '2020-09-30', '2020-11-30', '2020-12-31']
Calculando indicadores (EBIT anualizado)...
✅ 2020 concluído: 1,878 linhas

Processando 2021
  Baixando ITR 2021... ✓
  Baixando DFP 2021... ✓
Filtrando setor financeiro...
  Setor financeiro: 0 empresas serão excluídas
Limpando dados...
Pivotando contas...
  486 emp

$PASS3.SA: possibly delisted; no price data found  (1mo 2019-12-16 00:00:00 -> 2026-01-15 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1576465200, endDate = 1768446000")


  200/258 (198 com dados)...
  250/258 (247 com dados)...
  Preços obtidos: 255 tickers

✅ market_cap preenchido: 8,218/14,197 (57.9%)
Calculando métricas fundamentalistas...
  Baixando Selic (BCB)...
  ⚠ Selic indisponível: 502 Server Error: Bad Gateway for url: https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial=14/05/2016
  Calculando CAGR de lucro e receita (5 anos)...
  CAGR 'll_anual_para_cagr': 552 empresas com dezembro
  CAGR 'll_anual_para_cagr': 359 pares válidos
  CAGR 'receita_anual_para_cagr': 552 empresas com dezembro
  CAGR 'receita_anual_para_cagr': 626 pares válidos
  ✅ Métricas calculadas

  Resumo no trimestre mais recente (2025-12-31, 414 empresas):
      Prejuízo: 118 (29%)
      P/L > 15: 60 (14%)
      DL/EBITDA > 3: 92 (22%)
      CAGR lucro < 0: 50 (12%)


In [49]:
from datetime import datetime, timedelta

# 1. Baixa Selic corrigida
inicio = (datetime.today() - timedelta(days=365 * 10)).strftime("%d/%m/%Y")
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial={inicio}"
df_bcb = pd.DataFrame(requests.get(url).json())
df_bcb["data"]  = pd.to_datetime(df_bcb["data"], format="%d/%m/%Y")
df_bcb["valor"] = pd.to_numeric(df_bcb["valor"], errors="coerce") / 100
selic = df_bcb.set_index("data")["valor"].sort_index()
print(f"Selic carregada: {selic.index.min().date()} → {selic.index.max().date()}")

# 2. Mapeia Selic para cada data do df_final
datas = pd.DatetimeIndex(df_final["DT_FIM_EXERC"].dropna().unique())
mapa_selic = pd.Series(
    [selic.asof(d) if d >= selic.index[0] else np.nan for d in datas],
    index=datas,
)
df_final["selic"] = (df_final["DT_FIM_EXERC"].map(mapa_selic) * 100).round(2)

# 3. Recalcula colunas dependentes
df_final["roe_vs_selic"] = (df_final["roe"] - df_final["selic"]).round(2)

Selic carregada: 2016-05-14 → 2026-06-17


In [60]:

df_final["ll_anual_para_cagr"]      = _anualizar_serie(df_final, "ll")
df_final["receita_anual_para_cagr"] = _anualizar_serie(df_final, "receita")

df_final["cagr_lucro_3a"]   = (_calcular_cagr(df_final, "ll_anual_para_cagr",      anos=3) * 100).round(2)
df_final["cagr_receita_3a"] = (_calcular_cagr(df_final, "receita_anual_para_cagr", anos=3) * 100).round(2)

df_final = df_final.drop(columns=["ll_anual_para_cagr", "receita_anual_para_cagr"], errors="ignore")

print(f"\nColunas adicionadas. Verificação:")
print(df_final[["DT_FIM_EXERC", "selic", "roe", "roe_vs_selic",
                "cagr_lucro_3a", "cagr_receita_3a"]].dropna(subset=["selic"]).head(5))

  'll_anual_para_cagr': 552 empresas com dezembro
  'll_anual_para_cagr': 868 CAGRs calculados
  'receita_anual_para_cagr': 552 empresas com dezembro
  'receita_anual_para_cagr': 1446 CAGRs calculados

Colunas adicionadas. Verificação:
  DT_FIM_EXERC  selic     roe  roe_vs_selic  cagr_lucro_3a  cagr_receita_3a
0   2019-01-01    6.5   52.87         46.37            NaN              NaN
1   2019-02-28    6.5   16.71         10.21            NaN              NaN
2   2019-03-31    6.5   97.01         90.51            NaN              NaN
3   2019-03-31    6.5  161.21        154.71            NaN              NaN
4   2019-03-31    6.5   54.66         48.16            NaN              NaN


In [61]:
df_final.to_csv("fund_tri_comp.csv", index= False)
df_final

,CNPJ_CIA,DENOM_CIA,ticker,DT_FIM_EXERC,DT_INI_EXERC,meses_periodo,receita,ll,ebit,ebit_anualizado,...,roe_vs_selic,cagr_lucro_5a,cagr_receita_5a,p_l,descartar_prejuizo,descartar_pl_alto,descartar_dl_ebitda,descartar_cagr_lucro,cagr_lucro_3a,cagr_receita_3a
0,07.857.093/0001-14,AURA MINERALS INC.,AURA33,2019-01-01,2019-01-01,1.0,8.983080e+08,1.042110e+08,1.281260e+08,5.125040e+08,...,46.37,NaN,NaN,8.16,False,False,False,False,NaN,NaN
1,64.904.295/0001-03,CAMIL ALIMENTOS S.A.,CAML3,2019-02-28,2018-03-01,12.0,4.748825e+09,3.623870e+08,3.819810e+08,3.819810e+08,...,10.21,NaN,NaN,4.83,False,False,False,False,NaN,NaN
2,42.278.473/0001-03,WIZ CO PARTICIPAÇÕES E CORRETAGEM DE SEGUROS S.A.,WIZC3,2019-03-31,2019-01-01,3.0,1.543690e+08,5.663000e+07,8.613200e+07,3.445280e+08,...,90.51,NaN,NaN,3.19,False,False,False,False,NaN,NaN
3,45.242.914/0001-05,C&A MODAS S.A.,CEAB3,2019-03-31,2019-01-01,3.0,1.041151e+09,7.514360e+08,6.108330e+08,2.443332e+09,...,154.71,NaN,NaN,1.52,False,False,False,False,NaN,NaN
4,59.105.999/0001-86,WHIRLPOOL S.A.,WHRL3,2019-03-31,2019-01-01,3.0,1.702525e+09,3.485500e+08,2.778290e+08,1.111316e+09,...,48.16,NaN,NaN,2.48,False,False,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14192,84.683.374/0001-49,TUPY S.A.,TUPY3,2025-12-31,2025-01-01,12.0,9.692948e+09,-6.545520e+08,-1.564810e+08,-1.564810e+08,...,-41.04,NaN,17.88,NaN,True,False,True,False,NaN,-1.62
14193,84.683.671/0001-94,WETZEL S.A.,MWET4,2025-12-31,2025-01-01,12.0,1.975610e+08,-2.857800e+07,-4.120000e+06,-4.120000e+06,...,-447.61,NaN,6.20,NaN,True,False,True,False,NaN,-12.31
14194,89.463.822/0001-12,LUPATECH S.A,LUPA3,2025-12-31,2025-01-01,12.0,5.205400e+07,-6.017600e+07,-6.187800e+07,-6.187800e+07,...,-95.81,NaN,-0.95,NaN,True,False,False,False,NaN,-22.17
14195,90.400.888/0001-42,BCO SANTANDER (BRASIL) S.A.,SANB11,2025-12-31,2025-01-01,12.0,1.624946e+11,1.296512e+10,1.672899e+10,1.672899e+10,...,94.83,-0.73,20.95,12.73,False,False,False,True,-3.3,12.14


In [62]:
def _baixar_setores() -> pd.DataFrame:
    url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"
    df  = pd.read_csv(url, sep=";", encoding="latin1", dtype=str)
    df.columns   = df.columns.str.strip()
    df["CNPJ_CIA"]   = df["CNPJ_CIA"].str.strip()
    df["SETOR_ATIV"] = df["SETOR_ATIV"].str.strip()
    return df[["CNPJ_CIA", "SETOR_ATIV"]].drop_duplicates("CNPJ_CIA")

In [63]:
# ── 1. Filtros de descarte (aplica sobre o trimestre desejado) ────────────
def selecionar_trimestre(df: pd.DataFrame, trimestre: str,
                         top_n: int = 5) -> pd.DataFrame:
    recorte = df[df["DT_FIM_EXERC"] == pd.Timestamp(trimestre)].copy()

    if recorte.empty:
        print(f"  ⚠ Nenhuma linha para o trimestre {trimestre}")
        return pd.DataFrame()

    # Passo 2 — Lucratividade
    recorte = recorte[recorte["ll_positivo"] == True]
    recorte = recorte[recorte["margem_liquida"] >= 13]
    recorte = recorte[recorte["roe_vs_selic"] >= -2]

    # Passo 3 — Crescimento (NaN = histórico insuficiente → mantém)
    recorte = recorte[recorte["cagr_lucro_5a"].isna() | (recorte["cagr_lucro_5a"] > 0)]

    # Passo 4 — Endividamento
    recorte = recorte[recorte["dl_ebitda"].isna() | (recorte["dl_ebitda"] <= 3)]

    # Passo 5 — Valuation
    recorte = recorte[recorte["p_l"].between(3, 15)]

    print(f"  Após filtros: {len(recorte)} empresas elegíveis")

    if recorte.empty:
        return pd.DataFrame()

    # Adiciona setor se não estiver presente
    if "SETOR_ATIV" not in recorte.columns:
        #from exportar_por_setor import _baixar_setores
        recorte = recorte.merge(_baixar_setores(), on="CNPJ_CIA", how="left")

    # Calcula rank_magic dentro do setor para este trimestre
    # (evita depender de coluna que pode não existir no df_final)
    recorte["_rank_roic_s"] = (
        recorte[recorte["roic"] > 0]["roic"]
        .groupby(recorte["SETOR_ATIV"]).rank(ascending=False, method="min")
    ).reindex(recorte.index)

    recorte["_rank_ev_s"] = (
        recorte[recorte["ev_ebit"] > 0]["ev_ebit"]
        .groupby(recorte["SETOR_ATIV"]).rank(ascending=True, method="min")
    ).reindex(recorte.index)

    recorte["_rank_magic_s"] = np.where(
        recorte["_rank_roic_s"].notna() & recorte["_rank_ev_s"].notna(),
        recorte["_rank_roic_s"] + recorte["_rank_ev_s"],
        np.nan,
    )

    # Top N por setor
    recorte = recorte.sort_values("_rank_magic_s", na_position="last")
    top = (
        recorte.groupby("SETOR_ATIV", group_keys=False)
               .apply(lambda g: g.head(top_n))
               .reset_index(drop=True)
    )

    cols = ["SETOR_ATIV", "DENOM_CIA", "ticker", "DT_FIM_EXERC",
            "roe", "margem_liquida", "cagr_lucro_5a", "cagr_receita_5a",
            "dl_ebitda", "p_l", "_rank_roic_s", "_rank_ev_s", "_rank_magic_s"]
    return top[[c for c in cols if c in top.columns]].rename(columns={
        "_rank_roic_s":  "rank_roic_setor",
        "_rank_ev_s":    "rank_ev_ebit_setor",
        "_rank_magic_s": "rank_magic_setor",
    })

# Exemplo: melhores do último trimestre disponível
ultimo = str(df_final["DT_FIM_EXERC"].max())[:10]
carteira = selecionar_trimestre(df_final, ultimo, top_n=10)
print(f"Carteira {ultimo}: {len(carteira)} ações selecionadas")
print(carteira.to_string(index=False))

  Após filtros: 35 empresas elegíveis
Carteira 2025-12-31: 35 ações selecionadas
                                             SETOR_ATIV                                         DENOM_CIA ticker DT_FIM_EXERC    roe  margem_liquida  cagr_lucro_5a  cagr_receita_5a  dl_ebitda   p_l  rank_roic_setor  rank_ev_ebit_setor  rank_magic_setor
             Construção Civil, Mat. Constr. e Decoração                SONDOTECNICA ENGENHARIA SOLOS S.A.  SOND5   2025-12-31  52.80           15.78          11.11            28.62  -0.682069  3.96              1.0                 1.0               2.0
                               Seguradoras e Corretoras WIZ CO PARTICIPAÇÕES E CORRETAGEM DE SEGUROS S.A.  WIZC3   2025-12-31  28.37           26.59          13.00            11.49  -0.022963  4.17              1.0                 1.0               2.0
Emp. Adm. Part. - Const. Civil, Mat. Const. e Decoração                           JHSF PARTICIPACOES S.A.  JHSF3   2025-12-31  26.18           53.71          23

/tmp/ipykernel_18044/2105066852.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(top_n))


In [64]:
def selecionar_trimestre(df, trimestre, top_n, modo):
    """
    modo='por_setor' : top N dentro de cada setor  → carteira diversificada
    modo='global'    : top N entre todas as empresas → concentrada nos melhores
    modo='1_por_setor': exatamente 1 por setor       → máxima diversificação
    """
    recorte = df[df["DT_FIM_EXERC"] == pd.Timestamp(trimestre)].copy()

    # Filtros fundamentalistas
    recorte = recorte[recorte["ll_positivo"] == True]
    recorte = recorte[recorte["margem_liquida"] >= 13]
    recorte = recorte[recorte["roe_vs_selic"]   >= -2]
    recorte = recorte[recorte["cagr_lucro_5a"].isna() | (recorte["cagr_lucro_5a"] > 0)]
    recorte = recorte[recorte["dl_ebitda"].isna()     | (recorte["dl_ebitda"] <= 3)]
    recorte = recorte[recorte["p_l"].between(3, 15)]

    print(f"  {trimestre}: {len(recorte)} empresas após filtros")
    if recorte.empty:
        return pd.DataFrame()
    
    # Adiciona setor se não estiver presente
    if "SETOR_ATIV" not in recorte.columns:
        #from exportar_por_setor import _baixar_setores
        recorte = recorte.merge(_baixar_setores(), on="CNPJ_CIA", how="left")
        
    # Rank magic dentro do subconjunto filtrado
    recorte["_r_roic"] = recorte["roic"].rank(ascending=False, method="min")
    recorte["_r_ev"]   = recorte["ev_ebit"].rank(ascending=True,  method="min")
    recorte["_magic"]  = recorte["_r_roic"] + recorte["_r_ev"]
    recorte = recorte.sort_values("_magic", na_position="last")

    if modo == "global":
        top = recorte.head(top_n)

    elif modo == "1_por_setor":
        top = recorte.drop_duplicates(subset="SETOR_ATIV", keep="first")

    else:  # por_setor (padrão)
        top = (
            recorte.groupby("SETOR_ATIV", group_keys=False)
                   .apply(lambda g: g.head(top_n))
                   .reset_index(drop=True)
        )

    cols = ["SETOR_ATIV", "DENOM_CIA", "ticker", "DT_FIM_EXERC",
            "roe", "margem_liquida", "cagr_lucro_5a", "dl_ebitda",
            "p_l", "_magic"]
    return top[[c for c in cols if c in top.columns]].rename(
        columns={"_magic": "rank_magic"}
    )

In [65]:
trimestres = sorted(df_final["DT_FIM_EXERC"].unique())
carteiras = pd.concat(
    [selecionar_trimestre(df_final, str(t)[:10], top_n=3, modo="por_setor") for t in trimestres],
    ignore_index=True
)
carteiras.to_csv("carteiras_historicas.csv", index=False, encoding="utf-8-sig")

  2019-01-01: 0 empresas após filtros
  2019-02-28: 0 empresas após filtros
  2019-03-31: 20 empresas após filtros


/tmp/ipykernel_18044/3278792426.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(top_n))


  2019-05-31: 0 empresas após filtros
  2019-06-30: 19 empresas após filtros


/tmp/ipykernel_18044/3278792426.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(top_n))


  2019-08-31: 0 empresas após filtros
  2019-09-30: 17 empresas após filtros


...

/tmp/ipykernel_18044/3278792426.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(top_n))


  2025-08-31: 0 empresas após filtros
  2025-09-30: 33 empresas após filtros


/tmp/ipykernel_18044/3278792426.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(top_n))


  2025-11-30: 0 empresas após filtros
  2025-12-31: 35 empresas após filtros


/tmp/ipykernel_18044/3278792426.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(top_n))


In [66]:
trimestres = sorted(df_final["DT_FIM_EXERC"].unique())
carteiras = pd.concat(
    [selecionar_trimestre(df_final, str(t)[:10], top_n=15, modo="global") for t in trimestres],
    ignore_index=True
)
carteiras.to_csv("global_carteiras_historicas.csv", index=False, encoding="utf-8-sig")

  2019-01-01: 0 empresas após filtros
  2019-02-28: 0 empresas após filtros
  2019-03-31: 20 empresas após filtros
  2019-05-31: 0 empresas após filtros
  2019-06-30: 19 empresas após filtros
  2019-08-31: 0 empresas após filtros
  2019-09-30: 17 empresas após filtros
  2019-11-30: 0 empresas após filtros
  2019-12-31: 21 empresas após filtros
  2020-02-29: 0 empresas após filtros
  2020-03-31: 13 empresas após filtros
  2020-05-31: 0 empresas após filtros
  2020-06-30: 19 empresas após filtros
  2020-08-31: 0 empresas após filtros
  2020-09-30: 26 empresas após filtros
  2020-11-30: 0 empresas após filtros
  2020-12-31: 35 empresas após filtros
  2021-02-28: 0 empresas após filtros
  2021-03-31: 35 empresas após filtros
  2021-05-31: 0 empresas após filtros
  2021-06-30: 31 empresas após filtros
  2021-08-31: 0 empresas após filtros
  2021-09-30: 36 empresas após filtros
  2021-11-30: 0 empresas após filtros
  2021-12-31: 46 empresas após filtros
  2022-02-28: 0 empresas após filtros
